In [0]:
# In a Databricks notebook
# %pip install duckdb
%pip install --force-reinstall duckdb==1.5.3
import duckdb

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install pyyaml

In [0]:
import os
import duckdb
import yaml

# Ler configuração do arquivo YAML
config_path = "/Workspace/Users/zhang489yuan@gmail.com/databricks/mother_duck_medallion/config.yaml"

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

TOKEN = config['motherduck']['token']
DB = config['motherduck']['database']

con = duckdb.connect(
    f"md:{DB}?motherduck_token={TOKEN}"
)

In [0]:
Exato, esse padrão morre em 1 TB — e o problema são as duas linhas, não só uma:

fetchdf() puxa tudo para a memória do driver como um único pandas. 1 TB não cabe.
spark.createDataFrame(pdf) é single-thread e serializa tudo pelo driver de novo. Mesmo que coubesse, seria absurdamente lento.

A regra para volume grande é: o dado nunca passa pelo driver como um objeto único. Você usa Parquet no object storage como ponte, e cada lado lê/escreve em paralelo.
MotherDuck → Databricks (a direção do seu código)
Deixe o próprio DuckDB exportar para o S3 em Parquet, e o Spark lê em paralelo pelos executors:
python# 1. No DuckDB/MotherDuck: credencial do bucket
con.execute("""
    CREATE SECRET (
        TYPE S3,
        KEY_ID 'AKIA...',
        SECRET 'xxxx',
        REGION 'us-east-1'
    );
""")

# 2. COPY direto para Parquet (faz streaming, não materializa em RAM)
con.execute("""
    COPY (SELECT * FROM my_data.mte.rais_ident_2024)
    TO 's3://meu-bucket/rais_ident_2024/'
    (FORMAT PARQUET, PARTITION_BY (uf))   -- particionar ajuda muito
""")
python# 3. No Databricks: leitura paralela, sem driver gargalo
spark_df = spark.read.parquet("s3://meu-bucket/rais_ident_2024/")
spark_df.count()   # agora é distribuído de verdade
O COPY ... TO faz streaming para disco/S3 em vez de carregar tudo na memória, então 1 TB é viável (ainda que o DuckDB seja single-node — ele aguenta porque não materializa tudo de uma vez).

In [0]:
pdf = con.execute("""
SELECT *
FROM my_data.mte.rais_ident_2024 limit 1000
""").fetchdf()
spark_df = spark.createDataFrame(pdf)
# spark_df.show()

In [0]:
display(spark_df)

In [0]:
# Criar schema se não existir
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.duck")

(spark_df.repartition("CD_MUNICIPIO")     # 1. reorganiza os dados NA MEMÓRIA, agrupando por UF
   .write                 # 2. inicia a escrita
   .mode("overwrite")     # 3. se a tabela já existe, sobrescreve
   .partitionBy("CD_MUNICIPIO")     # 4. grava em PASTAS separadas por UF no disco
   .saveAsTable("workspace.duck.silver"))    # 5. registra como tabela no Unity Catalog

In [0]:
# Check what catalogs and schemas exist
print("Available catalogs:")
catalogs = spark.sql("SHOW CATALOGS").collect()
for cat in catalogs:
    print(f"  - {cat.catalog}")

print("\nSchemas in 'workspace' catalog:")
try:
    schemas = spark.sql("SHOW SCHEMAS IN workspace").collect()
    for schema in schemas:
        print(f"  - {schema.databaseName}")
except Exception as e:
    print(f"  Error: {e}")
    
print("\nCurrent catalog and schema:")
print(f"  Catalog: {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"  Schema: {spark.sql('SELECT current_schema()').collect()[0][0]}")